### Scrape Jobs

Python Job Scraping library "JobSpy": https://github.com/speedyapply/JobSpy

In [1]:
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

jobs = scrape_jobs(
    site_name=["indeed", "linkedin"],  # zip_recruiter, glassdoor, google, bayt, bdjobs
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=240,
    results_wanted=10,
)

df = pd.DataFrame(jobs)
print(f"Total jobs scraped: {len(df)}")
print(df[["title", "company", "site"]])

2026-04-06 10:30:53,082 - INFO - JobSpy:Linkedin - finished scraping


Total jobs scraped: 20
                                                title  \
0   Applied AI Engineer Co-op – Fall 2026 (Septemb...   
1   Product Manager / Product Owner, AI-Native Con...   
2                               Senior ML/AI Engineer   
3                                       Data Engineer   
4      AI (Artificial Intelligence) Engineer (Remote)   
5                                         AI Engineer   
6                   Lead AI Engineer (AI Foundations)   
7                 Oracle AI Engineer - Senior Manager   
8                 Oracle AI Engineer - Senior Manager   
9                 Oracle AI Engineer - Senior Manager   
10                              Senior ML/AI Engineer   
11                                        AI Engineer   
12                        AI Engineer - AI Department   
13                                        AI Engineer   
14                                        AI Engineer   
15                                 Python AI Engineer   
16      

### Filter

In [3]:
# We ensure the title contains BOTH "AI" and "Engineer"
# We need to do this, because for indeed, also the job descriptions are searched,
# so we could get job that only have "AI Engineer" in the description, but not in the title.
mask = df["title"].str.contains("AI", case=False, na=False) & df["title"].str.contains(
    "Engineer", case=False, na=False
)
matching_jobs = df[mask].copy()

# We keep only jobs that have the fields we want to store later
required_columns = ["title", "job_url", "description"]
has_required_values = (
    matching_jobs[required_columns]
    .fillna("")
    .apply(lambda column: column.astype(str).str.strip() != "")
    .all(axis=1)
)
complete_jobs = matching_jobs[has_required_values].copy()

# We drop duplicates based on the combination of 'title' and 'company'
filtered_jobs = complete_jobs.drop_duplicates(subset=["title", "company"]).copy()

print(f"Total jobs found: {len(df)}")
print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(f"Jobs with title, job URL, and description: {len(complete_jobs)}")
print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")

Total jobs found: 20
Jobs matching 'AI' + 'Engineer' in title: 18
Jobs with title, job URL, and description: 18
Jobs after deduplicating identical title/company pairs: 15


### Save the jobs as JSONL file

In [4]:
# Save the filtered jobs to a JSONL file in the "jobs" directory
# https://jsonlines.org/
output_dir = Path("jobs")
jsonl_path = output_dir / "1-scraped_jobs.jsonl"
jobs_to_save = filtered_jobs[["title", "job_url", "description"]].copy()
jobs_to_save.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

### Display jobs

In [5]:
results_to_show = filtered_jobs[["title", "company", "job_url"]].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

title,company,job_url
Applied AI Engineer Co-op – Fall 2026 (September Start),GE Aerospace,https://www.indeed.com/viewjob?jk=09374898ab564f0c
Senior ML/AI Engineer,SANDBAR,https://www.indeed.com/viewjob?jk=c146335910a915d7
AI (Artificial Intelligence) Engineer (Remote),"NexGen Technologies, Inc.",https://www.indeed.com/viewjob?jk=b8b25b4aaf482123
AI Engineer,NaN,https://www.indeed.com/viewjob?jk=c3d5ae2f3753dbff
Lead AI Engineer (AI Foundations),Capital One,https://www.indeed.com/viewjob?jk=85bdd2fdebefde9a
Oracle AI Engineer - Senior Manager,PwC,https://www.indeed.com/viewjob?jk=479942b34465086e
Senior ML/AI Engineer,Sandbar,https://www.linkedin.com/jobs/view/4397701205
AI Engineer,H2O.ai,https://www.linkedin.com/jobs/view/4392095303
AI Engineer - AI Department,Petlibro,https://www.linkedin.com/jobs/view/4396463746
AI Engineer,SoFi,https://www.linkedin.com/jobs/view/4393586304
